# Sentence Embeddings

There are several ways to get a vector for a sentence. Each approach has advantages and shortcomings. Choosing one depends on the task you want to perform with your vectors.

The first and very Naive approach will be to a simply average the vectors from word2vec. According to [Le and Mikolov](https://cs.stanford.edu/~quocle/paragraph_vector.pdf), this approach performs poorly for sentiment analysis tasks, because it "loses the word order in the same way as the standard bag-of-words models do" and "fails to recognize many sophisticated linguistic phenomena, for instance sarcasm". On the other hand, according to [Kenter et al.](http://aclweb.org/anthology/P/P16/P16-1089.pdf) 2016, "simply averaging word embeddings of all words in a text has proven to be a strong baseline or feature across a multitude of tasks", such as short text similarity tasks. 

## Averaging Word Vectors to obtain embeddings for the Sentence.

![Average Word Vectors](https://i.ibb.co/j86039L/average-word-vectors.png)

In [1]:
!pip install wget

  Created wheel for wget: filename=wget-3.2-cp36-none-any.whl size=9681 sha256=610a9a27b735355f84dc7cae1a446c5f41ee9d2a78399cc5820f8abf0c385cea
  Stored in directory: /root/.cache/pip/wheels/40/15/30/7d8f7cea2902b4db79e3fea550d7d7b85ecb27ef992b618f3f
Successfully built wget


In [2]:
import wget
wget.download('https://raw.githubusercontent.com/mxw/grmr/master/src/finaltests/bible.txt')

'bible.txt'

In [0]:
with open('bible.txt', 'r') as f:
  bible= f.read()

In [4]:
print(bible[:1000])

1:1 In the beginning God created the heaven and the earth.

1:2 And the earth was without form, and void; and darkness was upon
the face of the deep. And the Spirit of God moved upon the face of the
waters.

1:3 And God said, Let there be light: and there was light.

1:4 And God saw the light, that it was good: and God divided the light
from the darkness.

1:5 And God called the light Day, and the darkness he called Night.
And the evening and the morning were the first day.

1:6 And God said, Let there be a firmament in the midst of the waters,
and let it divide the waters from the waters.

1:7 And God made the firmament, and divided the waters which were
under the firmament from the waters which were above the firmament:
and it was so.

1:8 And God called the firmament Heaven. And the evening and the
morning were the second day.

1:9 And God said, Let the waters under the heaven be gathered together
unto one place, and let the dry land appear: and it was so.

1:10 And God called the d

In [0]:
# Cleaning the text
import re
cleaned= re.sub('[0-9][0-9]?:[0-9][0-9]? ',"", bible)

In [6]:
print(cleaned[:1000])

In the beginning God created the heaven and the earth.

And the earth was without form, and void; and darkness was upon
the face of the deep. And the Spirit of God moved upon the face of the
waters.

And God said, Let there be light: and there was light.

And God saw the light, that it was good: and God divided the light
from the darkness.

And God called the light Day, and the darkness he called Night.
And the evening and the morning were the first day.

And God said, Let there be a firmament in the midst of the waters,
and let it divide the waters from the waters.

And God made the firmament, and divided the waters which were
under the firmament from the waters which were above the firmament:
and it was so.

And God called the firmament Heaven. And the evening and the
morning were the second day.

And God said, Let the waters under the heaven be gathered together
unto one place, and let the dry land appear: and it was so.

And God called the dry land Earth; and the gathering together

In [7]:
import nltk
nltk.download('all',quiet= True)

True

In [0]:
from nltk.tokenize import word_tokenize, sent_tokenize
sentences= sent_tokenize(cleaned)

In [0]:
processed_data= []
for sent in sentences:
  processed_data.append(word_tokenize(sent))

In [0]:
# Training on first 10 sentences just for the sake visualising word vectors 
from gensim.models import Word2Vec
Model= Word2Vec(processed_data, size=300, sg=0) # Continuous Bag of Words (CBOW) Model, sg=0

In [11]:
len(Model.wv.vectors)

5695

In [0]:
import numpy as np
def document_vector(word2vec_model, doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc if word in word2vec_model.wv.vocab]
    return np.mean(word2vec_model[doc], axis=0)

In [13]:
X= list()
for doc in processed_data: 
    X.append(document_vector(Model, doc))

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:5: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  """


In [14]:
# Length of each sentence vector is 300
len(X[0])

300

In [15]:
len(sentences), len(X)

(29928, 29928)

*So, Now we have vectors for each sentence.*

## Doc2Vec

*The goal of doc2vec is to create a numeric representation of a document, regardless of its length. But unlike words, documents do not come in logical structures such as words, so the another method has to be found.*

*Doc2vec model is very simple. Just use a word2vec model, and add another vector (Paragraph ID below), like so:*

![](https://miro.medium.com/max/486/0*x-gtU4UlO8FAsRvL.)

**Distributed Memory Model of Paragraph Vectors (PV-DM)**

*Both paragraph vectors and word vectors are initialized randomly. Every paragraph vector is assigned to single document while word vectors are shared among all documents. Either averaging or concatenating both paragraph vector and words vectors and passing to stochastic gradient descent and the gradient is obtained via back propagation.*

Another approach is
**Distributed Bag of Words version of Paragraph Vector (PV-DBOW)**

*Another approach goes a different way. Instead of predicting next word, it use a paragraph vector to classify entire words in the document. During training, sampling a list of word and then form a classifer to classify whether word belongs to the document such that word vectors can be learnt.*

![](https://miro.medium.com/max/440/0*NtIsrbd4VQzUKVKr.)


In [0]:
import gensim
from gensim.models.doc2vec import TaggedDocument

In [0]:
tagged_corpus= [TaggedDocument(sentence, [i]) for i,sentence in enumerate(processed_data)]

In [18]:
tagged_corpus[1]

TaggedDocument(words=['And', 'the', 'earth', 'was', 'without', 'form', ',', 'and', 'void', ';', 'and', 'darkness', 'was', 'upon', 'the', 'face', 'of', 'the', 'deep', '.'], tags=[1])

In [19]:
model = gensim.models.Doc2Vec(tagged_corpus, dm= 0, alpha= 0.025, size = 20, min_alpha= 0.025, min_count= 0)

/usr/local/lib/python3.6/dist-packages/gensim/models/doc2vec.py:570: UserWarning: The parameter `size` is deprecated, will be removed in 4.0.0, use `vector_size` instead.
  warnings.warn("The parameter `size` is deprecated, will be removed in 4.0.0, use `vector_size` instead.")


In [0]:
model.train(tagged_corpus, total_examples= model.corpus_count, epochs=6)

In [21]:
print(processed_data[0])

['In', 'the', 'beginning', 'God', 'created', 'the', 'heaven', 'and', 'the', 'earth', '.']


In [22]:
vector = model.infer_vector(processed_data[0])
print(vector)

[-0.45603848 -0.37592176 -0.05948185 -0.36257693  0.03050764  0.11871831
 -0.22064203 -0.38281894  0.3343093  -0.1960553  -0.40976232 -0.01478472
 -0.24519458 -0.22442509 -0.32245943 -0.25224793 -0.6408372   0.22524743
 -0.2805425   0.2569309 ]


In [23]:
len(vector)

20

## Skip Thoughts 
*Skip Thoughts is a encoder decoder model that tries to reconstruct the surrounding sentences of an encoded passage. Sentences that share semantic and syntactic properties are thus mapped to similar vector representations.*

In [25]:
!pip install skipthoughts

### Available pretrained models
**UniSkip**
It uses the nn.GRU layer from torch with the cudnn backend. It is the fastest implementation, but the dropout is sampled after each time-step in the cudnn implementation... (equals bad regularization)

**DropUniSkip**
It uses the nn.GRUCell layer from torch with the cudnn backend. It is slightly slower than UniSkip, however the dropout is sampled once for all time-steps in a sequence (good regularization).

**BayesianUniSkip**
It uses a custom GRU layer with a torch backend. It is at least two times slower than UniSkip, however the dropout is sampled once for all time-steps for each Linear (best regularization).

**BiSkip**
Equivalent to UniSkip, but with a bi-sequential GRU.

In [0]:
import torch
from torch.autograd import Variable
import sys
sys.path.append('skip-thoughts.torch/pytorch')
from skipthoughts import UniSkip

In [27]:
dir_st = 'data/skip-thoughts'
vocab = ['robots', 'are', 'very', 'cool', '<eos>', 'BiDiBu']
uniskip = UniSkip(dir_st, vocab)

input = Variable(torch.LongTensor([
    [1,2,3,4,0], # robots are very cool 0
    [6,2,3,4,5]  # bidibu are very cool <eos>
])) # <eos> token is optional

/usr/local/lib/python3.6/dist-packages/torch/nn/modules/rnn.py:51: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.25 and num_layers=1
  "num_layers={}".format(dropout, num_layers))


In [28]:
output_seq2vec = uniskip(input, lengths=[4,5])
print(output_seq2vec.size()) # batch_size x 2400

torch.Size([2, 2400])


In [29]:
output_seq2seq = uniskip(input)
print(output_seq2seq.size()) # batch_size x seq_len x 2400

torch.Size([2, 2400])


In [30]:
output_seq2seq

tensor([[ 0.1793, -0.1286,  0.0964,  ...,  0.1034, -0.5499, -0.1072],
        [ 0.0705, -0.0631,  0.0190,  ..., -0.0402, -0.3044, -0.0416]],
       grad_fn=<ViewBackward>)

---